# AI Readiness Scorecard — Microsoft Fabric

**Platform:** Microsoft Fabric Spark runtime  
**Purpose:** Example assessment of data conditions that may affect a proposed analytics or AI use case.

## Assessment dimensions

This example combines five dimensions into a weighted score. The weights and thresholds are illustrative; agree on definitions and adjust them for each domain and use case before relying on the results.

| Dimension | Example weight | What it considers |
|---|---:|---|
| Data quality | 30% | Completeness, uniqueness, and validity checks |
| Data governance | 25% | Classification, lineage, and ownership evidence |
| Semantic consistency | 20% | Shared definitions and documented business rules |
| Volume and representativeness | 15% | Whether available data is suitable for the proposed analysis |
| Freshness and latency | 10% | Whether update frequency meets the use case's needs |

The result is a screening aid, not a certification or approval to use data with a model. Review the underlying evidence and involve data owners, security, privacy, and other required reviewers before making a production decision.

The example configuration refers to sample domain and table names. Update it to match the attached Lakehouse and validate each scoring rule before running the notebook.


In [ ]:
# ============================================================
# CELL 1: Imports and Framework Configuration
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, FloatType, 
    IntegerType, TimestampType, BooleanType
)
from delta.tables import DeltaTable
from datetime import datetime, timezone
import math

# ── Scoring Weights ────────────────────────────────────────
WEIGHTS = {
    "data_quality":          0.30,
    "data_governance":       0.25,
    "semantic_consistency":  0.20,
    "volume_representativeness": 0.15,
    "freshness_latency":     0.10,
}
assert abs(sum(WEIGHTS.values()) - 1.0) < 0.001, "Weights must sum to 1.0"

# ── Readiness Thresholds ───────────────────────────────────
READINESS_TIERS = {
    (0.80, 1.01): ("Higher readiness",        "Green",  "Review against the proposed use case and its controls."),
    (0.60, 0.80): ("Readiness with conditions",  "Yellow", "Documented gaps need owners and a remediation plan."),
    (0.40, 0.60): ("In development",  "Orange", "Further data or governance work is needed."),
    (0.00, 0.40): ("Foundational work needed",    "Red",    "Address foundational issues before reassessing."),
}

def get_readiness_tier(score: float):
    for (low, high), (tier, color, description) in READINESS_TIERS.items():
        if low <= score < high:
            return tier, color, description
    return "Unknown", "Grey", ""

# ── Domain Configuration ───────────────────────────────────
# Customise this for each deployment
DOMAINS = ["Customer", "Finance", "Product", "Operations", "HR"]

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
RESULTS_TABLE = "ai_readiness_scores"

print(f"AI Readiness Scorecard | Run: {RUN_ID}")
print(f"Domains: {', '.join(DOMAINS)}")

In [ ]:
# ============================================================
# CELL 2: Dimension 1 — Data Quality Score
# Pull from previous quality assessment results
# ============================================================

def get_data_quality_score(domain: str) -> dict:
    """
    Retrieves the most recent quality assessment scores for a domain
    from the dq_assessment_results table (produced by notebook 01).
    
    Returns:
        dict with score (0–1), component details, and notes
    """
    try:
        # Get latest run for this domain
        latest_run = spark.sql(f"""
            SELECT run_id
            FROM dq_assessment_results
            WHERE domain = '{domain}'
            ORDER BY assessment_ts DESC
            LIMIT 1
        """).collect()
        
        if not latest_run:
            return {"score": 0.0, "notes": "No quality assessment data found. Run notebook 01 first.",
                    "components": {}}
        
        latest_run_id = latest_run[0]["run_id"]
        
        domain_quality = spark.sql(f"""
            SELECT
                AVG(completeness_score) AS avg_completeness,
                AVG(uniqueness_score)   AS avg_uniqueness,
                AVG(freshness_score)    AS avg_freshness,
                AVG(overall_score)      AS avg_overall,
                COUNT(*)                AS table_count,
                SUM(CASE WHEN quality_tier IN ('CRITICAL','POOR','ERROR') THEN 1 ELSE 0 END) AS critical_tables
            FROM dq_assessment_results
            WHERE domain = '{domain}' AND run_id = '{latest_run_id}'
        """).collect()[0]
        
        raw_score = float(domain_quality["avg_overall"] or 0)
        critical_penalty = (domain_quality["critical_tables"] or 0) * 0.05  # 5% penalty per critical table
        adjusted_score = max(0.0, raw_score - critical_penalty)
        
        return {
            "score": round(adjusted_score, 4),
            "components": {
                "avg_completeness": float(domain_quality["avg_completeness"] or 0),
                "avg_uniqueness":   float(domain_quality["avg_uniqueness"] or 0),
                "avg_freshness":    float(domain_quality["avg_freshness"] or 0),
                "table_count":      int(domain_quality["table_count"] or 0),
                "critical_tables":  int(domain_quality["critical_tables"] or 0),
            },
            "notes": f"{int(domain_quality['table_count'])} tables; {int(domain_quality['critical_tables'] or 0)} critical"
        }
    
    except Exception as e:
        return {"score": 0.0, "notes": f"Error: {e}", "components": {}}

print("Data quality score function defined.")

In [ ]:
# ============================================================
# CELL 3: Dimension 2 — Data Governance Score
# Checks: sensitivity labels, lineage coverage, documented ownership
# ============================================================

def get_governance_score(domain: str) -> dict:
    """
    Scores governance maturity for a domain based on:
    1. Sensitivity label coverage (Purview integration)
    2. Lineage coverage (% of tables with tracked lineage)
    3. Data ownership documentation
    4. Access control review recency
    """
    try:
        governance_data = spark.sql(f"""
            SELECT
                COUNT(*) AS total_tables,
                SUM(CASE WHEN has_sensitivity_label = true  THEN 1 ELSE 0 END) AS labeled_tables,
                SUM(CASE WHEN has_lineage_tracked  = true   THEN 1 ELSE 0 END) AS lineage_tables,
                SUM(CASE WHEN has_documented_owner = true   THEN 1 ELSE 0 END) AS owned_tables,
                SUM(CASE WHEN access_review_current = true  THEN 1 ELSE 0 END) AS reviewed_tables
            FROM platform_services.governance_registry
            WHERE data_domain = '{domain}'
        """).collect()[0]
        
        total = int(governance_data["total_tables"] or 1)
        
        label_rate    = (governance_data["labeled_tables"]  or 0) / total
        lineage_rate  = (governance_data["lineage_tables"]  or 0) / total
        owner_rate    = (governance_data["owned_tables"]    or 0) / total
        review_rate   = (governance_data["reviewed_tables"] or 0) / total
        
        # Governance score is average of the four sub-dimensions
        score = (label_rate + lineage_rate + owner_rate + review_rate) / 4
        
        return {
            "score": round(score, 4),
            "components": {
                "sensitivity_label_rate": round(label_rate, 4),
                "lineage_coverage_rate":  round(lineage_rate, 4),
                "ownership_rate":         round(owner_rate, 4),
                "access_review_rate":     round(review_rate, 4),
            },
            "notes": f"{total} tables; label: {label_rate:.0%}, lineage: {lineage_rate:.0%}"
        }
    
    except Exception as e:
        # Governance registry may not be populated yet — return baseline score
        return {
            "score": 0.3,  # Default: partial governance assumed
            "notes": f"Governance registry not found — using baseline score. Deploy registry to get accurate scoring.",
            "components": {}
        }

print("Governance score function defined.")

In [ ]:
# ============================================================
# CELL 4: Dimension 3 — Semantic Consistency Score
# Checks: semantic model coverage, certified measure count,
#         data dictionary completeness
# ============================================================

def get_semantic_score(domain: str) -> dict:
    """
    Scores semantic consistency for a domain:
    1. Does the domain have a certified semantic model?
    2. Are key measures documented with business definitions?
    3. Is the domain's business glossary complete?
    """
    try:
        semantic_data = spark.sql(f"""
            SELECT
                has_certified_semantic_model,
                certified_measure_count,
                total_measure_count,
                business_glossary_completeness_pct,
                column_description_coverage_pct
            FROM platform_services.semantic_registry
            WHERE data_domain = '{domain}'
            ORDER BY last_updated DESC
            LIMIT 1
        """).collect()
        
        if not semantic_data:
            return {"score": 0.2, "notes": "No semantic model registered.", "components": {}}
        
        row = semantic_data[0]
        
        model_score     = 1.0 if row["has_certified_semantic_model"] else 0.0
        total_measures  = row["total_measure_count"] or 1
        measure_score   = (row["certified_measure_count"] or 0) / total_measures
        glossary_score  = (row["business_glossary_completeness_pct"] or 0) / 100
        desc_score      = (row["column_description_coverage_pct"] or 0) / 100
        
        score = (model_score * 0.4) + (measure_score * 0.3) + (glossary_score * 0.2) + (desc_score * 0.1)
        
        return {
            "score": round(score, 4),
            "components": {
                "has_certified_model":    bool(row["has_certified_semantic_model"]),
                "measure_doc_rate":       round(measure_score, 4),
                "glossary_completeness":  round(glossary_score, 4),
                "column_desc_coverage":   round(desc_score, 4),
            },
            "notes": f"Model: {'present' if row['has_certified_semantic_model'] else 'not present'}; measures: {measure_score:.0%} documented"
        }
    
    except Exception as e:
        return {"score": 0.2, "notes": f"Semantic registry error: {e}", "components": {}}

print("Semantic score function defined.")

In [ ]:
# ============================================================
# CELL 5: Dimension 4 & 5 — Volume and Freshness Scores
# ============================================================

def get_volume_score(domain: str) -> dict:
    """
    Scores volume and representativeness.
    Checks whether Gold tables in the domain have sufficient records
    for AI model training / inference.
    """
    try:
        gold_tables = [t.name for t in spark.catalog.listTables() 
                       if t.name.startswith(f"gold_") 
                       and DOMAIN_MAP.get(t.name) == domain]
        
        if not gold_tables:
            return {"score": 0.0, "notes": "No Gold tables found for domain.", "components": {}}
        
        scores = []
        details = {}
        
        for table in gold_tables:
            count = spark.table(table).count()
            details[table] = count
            
            # Scoring curve: 10k records = 0.5; 100k = 0.8; 1M+ = 1.0
            if count >= 1_000_000:
                scores.append(1.0)
            elif count >= 100_000:
                scores.append(0.8 + (count - 100_000) / (1_000_000 - 100_000) * 0.2)
            elif count >= 10_000:
                scores.append(0.5 + (count - 10_000) / (100_000 - 10_000) * 0.3)
            elif count >= 1_000:
                scores.append(0.2 + (count - 1_000) / (10_000 - 1_000) * 0.3)
            else:
                scores.append(count / 1_000 * 0.2)  # < 1k: linearly up to 0.2
        
        avg_score = sum(scores) / len(scores)
        
        return {
            "score": round(avg_score, 4),
            "components": details,
            "notes": f"{len(gold_tables)} Gold tables; avg volume score: {avg_score:.1%}"
        }
    
    except Exception as e:
        return {"score": 0.3, "notes": f"Volume check error: {e}", "components": {}}


def get_freshness_score(domain: str) -> dict:
    """
    Retrieves freshness scores from the DQ assessment results.
    Maps domain freshness to AI use case requirements.
    """
    try:
        freshness_data = spark.sql(f"""
            SELECT AVG(freshness_score) AS avg_freshness
            FROM dq_assessment_results
            WHERE domain = '{domain}'
              AND run_id = (
                SELECT MAX(run_id) FROM dq_assessment_results WHERE domain = '{domain}'
              )
        """).collect()[0]
        
        score = float(freshness_data["avg_freshness"] or 0)
        
        return {
            "score": round(score, 4),
            "notes": f"Avg freshness vs SLA: {score:.1%}",
            "components": {}
        }
    
    except Exception as e:
        return {"score": 0.5, "notes": f"Freshness data unavailable: {e}", "components": {}}

print("Volume and freshness score functions defined.")

In [ ]:
# ============================================================
# CELL 6: Calculate AI Readiness Score per Domain
# ============================================================

import json

# Domain map for volume checks (reverse lookup)
DOMAIN_MAP = {v: k for k, v in {
    "bronze_customers":    "Customer",
    "bronze_orders":       "Finance",
    "bronze_transactions": "Finance",
    "bronze_products":     "Product",
    "bronze_employees":    "HR",
    "bronze_support":      "Operations",
    "bronze_events":       "Customer",
}.items()}

scorecard_results = []

for domain in DOMAINS:
    print(f"\nScoring domain: {domain}")
    
    # Get dimension scores
    dq   = get_data_quality_score(domain)
    gov  = get_governance_score(domain)
    sem  = get_semantic_score(domain)
    vol  = get_volume_score(domain)
    fresh = get_freshness_score(domain)
    
    # Weighted composite score
    composite = (
        dq["score"]    * WEIGHTS["data_quality"]              +
        gov["score"]   * WEIGHTS["data_governance"]           +
        sem["score"]   * WEIGHTS["semantic_consistency"]      +
        vol["score"]   * WEIGHTS["volume_representativeness"] +
        fresh["score"] * WEIGHTS["freshness_latency"]
    )
    
    tier, color, description = get_readiness_tier(composite)
    
    result = {
        "run_id":               RUN_ID,
        "scored_at":            datetime.now(timezone.utc),
        "domain":               domain,
        "data_quality_score":   dq["score"],
        "governance_score":     gov["score"],
        "semantic_score":       sem["score"],
        "volume_score":         vol["score"],
        "freshness_score":      fresh["score"],
        "composite_score":      round(composite, 4),
        "readiness_tier":       tier,
        "readiness_color":      color,
        "tier_description":     description,
        "dq_notes":             dq["notes"],
        "governance_notes":     gov["notes"],
        "semantic_notes":       sem["notes"],
        "volume_notes":         vol["notes"],
        "freshness_notes":      fresh["notes"],
    }
    
    scorecard_results.append(result)
    
    print(f"  {domain}: {composite:.1%} — {tier}")
    print(f"     Quality: {dq['score']:.0%}  |  Governance: {gov['score']:.0%}  |  "
          f"Semantic: {sem['score']:.0%}  |  Volume: {vol['score']:.0%}  |  Freshness: {fresh['score']:.0%}")

print("\nScoring complete.")

In [ ]:
# ============================================================
# CELL 7: Write Scorecard Results to Delta Table
# ============================================================

schema = StructType([
    StructField("run_id",              StringType(),    False),
    StructField("scored_at",           TimestampType(), False),
    StructField("domain",              StringType(),    False),
    StructField("data_quality_score",  FloatType(),     True),
    StructField("governance_score",    FloatType(),     True),
    StructField("semantic_score",      FloatType(),     True),
    StructField("volume_score",        FloatType(),     True),
    StructField("freshness_score",     FloatType(),     True),
    StructField("composite_score",     FloatType(),     False),
    StructField("readiness_tier",      StringType(),    False),
    StructField("readiness_color",     StringType(),    False),
    StructField("tier_description",    StringType(),    True),
    StructField("dq_notes",            StringType(),    True),
    StructField("governance_notes",    StringType(),    True),
    StructField("semantic_notes",      StringType(),    True),
    StructField("volume_notes",        StringType(),    True),
    StructField("freshness_notes",     StringType(),    True),
])

scorecard_df = spark.createDataFrame(scorecard_results, schema=schema)

scorecard_df.write.format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable(RESULTS_TABLE)

print(f"Scorecard written to Tables/{RESULTS_TABLE}")

In [ ]:
# ============================================================
# CELL 8: Executive Scorecard Output
# This is the output designed for executive briefing use
# ============================================================

print()
print("═" * 70)
print("  AI READINESS SCORECARD")
print(f"  Assessment Date: {datetime.now(timezone.utc).strftime('%B %d, %Y')}")
print("═" * 70)
print()

# Sort by composite score descending
sorted_results = sorted(scorecard_results, key=lambda r: r["composite_score"], reverse=True)

print(f"  {'DOMAIN':<16} {'SCORE':>7}  {'TIER':<20}  BREAKDOWN")
print(f"  {'─'*14:<16} {'─'*5:>7}  {'─'*18:<20}  {'─'*35}")

for r in sorted_results:
    breakdown = (f"Q:{r['data_quality_score']:.0%} G:{r['governance_score']:.0%} "
                 f"S:{r['semantic_score']:.0%} V:{r['volume_score']:.0%} F:{r['freshness_score']:.0%}")
    print(f"  {r['domain']:<14} {r['composite_score']:>6.1%}  {r['readiness_tier']:<26}  {breakdown}")

print()
print("  Legend: Q=Quality  G=Governance  S=Semantic  V=Volume  F=Freshness")
print()

# Summary stats
green_domains  = [r for r in sorted_results if r["readiness_color"] == "Green"]
yellow_domains = [r for r in sorted_results if r["readiness_color"] == "Yellow"]
orange_domains = [r for r in sorted_results if r["readiness_color"] == "Orange"]
red_domains    = [r for r in sorted_results if r["readiness_color"] == "Red"]

print(f"  Higher readiness ({len(green_domains)}): {', '.join(r['domain'] for r in green_domains) or 'None'}")
print(f"  Readiness with conditions ({len(yellow_domains)}): {', '.join(r['domain'] for r in yellow_domains) or 'None'}")
print(f"  In development ({len(orange_domains)}): {', '.join(r['domain'] for r in orange_domains) or 'None'}")
print(f"  Foundational work needed ({len(red_domains)}): {', '.join(r['domain'] for r in red_domains) or 'None'}")
print()

if green_domains:
    print(f"  Suggested next step: review {green_domains[0]['domain']} with its data owner and use-case reviewers.")

if red_domains:
    print(f"  Suggested follow-up: confirm the highest-priority gaps for {red_domains[-1]['domain']} with its owner.")

print()
print("═" * 70)
print(f"  Full results in: Tables/{RESULTS_TABLE}")
print("═" * 70)